# Crop Condition Signal Research
### Quantitative Research - Agriculture Commodity Signals

**Objective:** Generate tradable signals from USDA weekly Crop Condition & Progress reports.

In [1]:
%load_ext autoreload
%autoreload 2

import pandas as pd
import numpy as np
from datetime import datetime
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import warnings
warnings.filterwarnings('ignore')

from my_holiday.cbt import CBT
from my_holiday.holiday_utils import apply_date_rules
from data.data_loader import PriceDataLoader
from data.data_loader3 import CropCnPDataLoader
from pages.other_reports.crop_condition_and_progress.signal_crop_cnp import SignalCropCnp

print('All imports successful')

All imports successful


---
## 1. Data Loading
Load crop condition signals, raw condition/progress data, and futures prices.

In [2]:
COMMODITIES = {
    'WINTER WHEAT':   {'price_cmd': 'W', 'contract_month': 'N', 'name': 'Wheat'}

}
MONTH_CODE = {1:'F',2:'G',3:'H',4:'J',5:'K',6:'M',7:'N',8:'Q',9:'U',10:'V',11:'X',12:'Z'}
FORWARD_HORIZONS = [5, 10, 20]
SEASON_PHASES = {
    'planting': [9, 10, 11],
    'dormancy': [12, 1, 2],
    'growing':  [3, 4, 5],
    'harvest':  [6, 7],
    'off_season': [8]  # Aug = post-harvest / pre-planting gap
}
signal_calc = SignalCropCnp(signal_config=None)
all_signals = {}
for cmd in COMMODITIES:
    print(f'Loading {cmd} signals...')
    df = signal_calc.calculate_signal_all(cmd=cmd)
    df.dropna(subset=['signal_value'], inplace=True)
    all_signals[cmd] = df
    print(f'  -> {len(df)} rows, signals: {df["signal_name"].unique().tolist()}')

Loading WINTER WHEAT signals...
{"asctime": "2026-04-09 17:13:53,415", "levelname": "INFO", "name": "app", "message": "Data loaded from cache for page: crop_condition_and_progress", "pathname": "c:\\Users\\yuhang.hou\\projects\\multipage_dash\\olam_dash\\data\\data_loader3.py", "lineno": 129, "process": 42748, "threadName": "MainThread"}
  -> 1290 rows, signals: ['crop_condition_normalized', 'crop_condition_normalized_diff', 'crop_condition_historical_surprise']


In [3]:
raw_loader = CropCnPDataLoader()
raw_condition, raw_progress = {}, {}

for cmd in COMMODITIES:
    print(f'\n--- {cmd} ---')
    raw_condition[cmd] = raw_loader.load_data(cmd, 'condition')
    print(f'  Condition: {len(raw_condition[cmd])} rows')
    try:
        raw_progress[cmd] = raw_loader.load_data(cmd, 'progress')
        print(f'  Progress: {len(raw_progress[cmd])} rows, types: {raw_progress[cmd]["type"].unique()}')
    except Exception as e:
        raw_progress[cmd] = None
        print(f'  Progress: Not available')


--- WINTER WHEAT ---
{"asctime": "2026-04-09 17:13:55,318", "levelname": "INFO", "name": "app", "message": "Data loaded from cache for page: crop_condition_and_progress", "pathname": "c:\\Users\\yuhang.hou\\projects\\multipage_dash\\olam_dash\\data\\data_loader3.py", "lineno": 129, "process": 42748, "threadName": "MainThread"}
  Condition: 78615 rows
{"asctime": "2026-04-09 17:13:55,323", "levelname": "INFO", "name": "app", "message": "Data loaded from cache for page: crop_condition_and_progress", "pathname": "c:\\Users\\yuhang.hou\\projects\\multipage_dash\\olam_dash\\data\\data_loader3.py", "lineno": 129, "process": 42748, "threadName": "MainThread"}
  Progress: 0 rows, types: []


In [4]:
price_data = {}
for cmd, info in COMMODITIES.items():
    print(f'Loading {info["name"]} price...')
    loader = PriceDataLoader('price_data', reload_freq=1)
    df = loader.load_data(info['price_cmd'], filter_holiday=True)
    df['date'] = pd.to_datetime(df['date'])
    df['month_code'] = df['expiry_month'].map(MONTH_CODE)
    df = df[df['month_code'] == info['contract_month']].reset_index(drop=True)
    df = df.sort_values(['date', 'expiry_year', 'expiry_month'])
    df = df.groupby('date').head(1).reset_index(drop=True)
    df['return'] = df['value'].pct_change()
    df['log_return'] = np.log(df['value'] / df['value'].shift(1))
    df['realized_vol_20d'] = df['log_return'].rolling(20, min_periods=10).std() * np.sqrt(252)
    price_data[cmd] = df
    print(f'  -> {len(df)} rows, {df["date"].min().date()} to {df["date"].max().date()}')

Loading Wheat price...
  -> 6843 rows, 1999-01-27 to 2026-04-08


---
## 2. Custom Signal Engineering
6 new signals: Condition Momentum, Acceleration, YoY Change, Extreme Flags, G+E Z-score, G+E Momentum

In [5]:
def build_custom_signals(condition_df, lookback_years=10):
    df = condition_df[condition_df['location'] == 'US TOTAL'].copy()
    df['date'] = pd.to_datetime(df['date'])
    pivoted = df.pivot(index='date', columns='type', values='value').reset_index()
    for col in ['excellent', 'good', 'fair', 'poor', 'very poor']:
        if col not in pivoted.columns: pivoted[col] = 0
        pivoted[col] = pivoted[col].fillna(0)
    weights = {'excellent': 1.25, 'good': 1.0, 'fair': 0.75, 'poor': 0.5, 'very poor': 0.0}
    pivoted['condition_index'] = sum(pivoted[c]*w for c,w in weights.items())
    pivoted['good_excellent_pct'] = pivoted['excellent'] + pivoted['good']
    pivoted = pivoted.sort_values('date').reset_index(drop=True)
    pivoted['year'] = pivoted['date'].dt.year
    pivoted['week'] = pivoted['date'].dt.isocalendar().week.astype(int)
    # Signal 1: Momentum
    pivoted['condition_change'] = pivoted['condition_index'].diff()
    pivoted['condition_momentum_3w'] = pivoted['condition_change'].rolling(3, min_periods=2).mean()
    # Signal 2: Acceleration
    pivoted['condition_acceleration'] = pivoted['condition_change'].diff()
    # Signal 3: YoY
    prev = pivoted[['week','year','condition_index']].copy()
    prev['year'] = prev['year'] - 1
    prev = prev.rename(columns={'condition_index':'prev_yr_index'})
    pivoted = pivoted.merge(prev[['week','year','prev_yr_index']], on=['week','year'], how='left')
    pivoted['yoy_condition_change'] = pivoted['condition_index'] - pivoted['prev_yr_index']
    # Signal 4: Extreme flags
    pivoted['condition_pct_rank'] = pivoted['condition_index'].rank(pct=True)
    pivoted['is_extreme_good'] = (pivoted['condition_pct_rank'] >= 0.8).astype(int)
    pivoted['is_extreme_bad'] = (pivoted['condition_pct_rank'] <= 0.2).astype(int)
    # Signal 5: G+E Z-score
    hist = pivoted.copy()
    def deseason_ge(row):
        sw = hist[(hist['week']==row['week']) & (hist['year']<row['year']) & (hist['year']>row['year']-lookback_years)]
        if len(sw)<3: return np.nan
        m,s = sw['good_excellent_pct'].mean(), sw['good_excellent_pct'].std()
        return (row['good_excellent_pct']-m)/s if s>0 and not pd.isna(s) else np.nan
    pivoted['ge_zscore'] = pivoted.apply(deseason_ge, axis=1)
    # Signal 6: G+E Momentum
    pivoted['ge_change'] = pivoted['good_excellent_pct'].diff()
    pivoted['ge_momentum_3w'] = pivoted['ge_change'].rolling(3, min_periods=2).mean()
    pivoted['signal_date'] = pivoted['date'].apply(lambda x: apply_date_rules(x, '+2B', 'CBT'))
    pivoted['signal_date'] = pd.to_datetime(pivoted['signal_date'])
    cols = ['signal_date','condition_momentum_3w','condition_acceleration','yoy_condition_change',
            'is_extreme_good','is_extreme_bad','ge_zscore','ge_momentum_3w','good_excellent_pct','condition_index']
    return pivoted[cols]

custom_signals = {}
for cmd in COMMODITIES:
    print(f'Building {cmd} custom signals...')
    custom_signals[cmd] = build_custom_signals(raw_condition[cmd])
    print(f'  -> {len(custom_signals[cmd])} rows')
    display(custom_signals[cmd].tail(3))
custom_signals[cmd].describe()

Building WINTER WHEAT custom signals...
  -> 505 rows


type,signal_date,condition_momentum_3w,condition_acceleration,yoy_condition_change,is_extreme_good,is_extreme_bad,ge_zscore,ge_momentum_3w,good_excellent_pct,condition_index
502,2025-11-18,-0.166667,-1.25,NaN,0,0,-0.463768,-1.333333,45,80.50
503,2025-11-25,0.666667,3.00,NaN,0,0,-0.133333,0.000000,48,82.75
504,2026-04-07,-3.250000,-13.50,NaN,0,0,-0.871366,-4.333333,35,71.50


type,signal_date,condition_momentum_3w,condition_acceleration,yoy_condition_change,is_extreme_good,is_extreme_bad,ge_zscore,ge_momentum_3w,good_excellent_pct,condition_index
count,505,503.000000,503.000000,457.000000,505.000000,505.000000,434.000000,503.000000,505.000000,505.000000
mean,2012-11-29 05:42:10.693069312,-0.010686,-0.021869,0.177243,0.201980,0.203960,0.051622,-0.016236,47.908911,79.793069
min,1999-11-02 00:00:00,-28.916667,-80.500000,-26.250000,0.000000,0.000000,-3.707627,-18.666667,0.000000,0.000000
25%,2006-05-23 00:00:00,-0.916667,-1.250000,-11.000000,0.000000,0.000000,-0.885906,-1.333333,40.000000,74.250000
50%,2012-11-20 00:00:00,-0.166667,0.000000,1.750000,0.000000,0.000000,0.087717,0.000000,48.000000,81.750000
75%,2019-05-29 00:00:00,0.458333,1.250000,8.250000,0.000000,0.000000,0.661918,1.000000,56.000000,86.500000
max,2026-04-07 00:00:00,23.666667,153.750000,30.500000,1.000000,1.000000,14.433757,13.000000,78.000000,99.250000
std,NaN,2.861693,9.905028,13.439029,0.401876,0.403339,1.382045,3.076495,11.436662,9.938389


---
## 3. Master Analysis Table
Merge all signals with forward returns at multiple horizons (1W, 2W, 4W).

In [6]:
def get_season(month):
    for phase, months in SEASON_PHASES.items():
        if month in months: return phase
    return 'unknown'

def build_master_table(cmd, signals_df, custom_df, price_df):
    sig = signals_df.pivot_table(index='signal_date', columns='signal_name', values='signal_value').reset_index()
    sig['signal_date'] = pd.to_datetime(sig['signal_date'])
    merged = pd.merge(sig, custom_df, on='signal_date', how='left')
    prc = price_df[['date','value']].copy().sort_values('date').reset_index(drop=True)
    for h in FORWARD_HORIZONS:
        prc[f'fwd_{h}d_ret'] = prc['value'].shift(-h) / prc['value'] - 1
    prc['date'] = pd.to_datetime(prc['date'])
    merged = pd.merge(merged, prc, left_on='signal_date', right_on='date', how='inner', suffixes=('','_price'))
    merged = merged.dropna(subset=['fwd_5d_ret'])
    merged['month'] = merged['signal_date'].dt.month
    merged['year'] = merged['signal_date'].dt.year
    merged['season'] = merged['month'].apply(get_season)
    vol_df = price_df[['date','realized_vol_20d']].copy()
    vol_df['date'] = pd.to_datetime(vol_df['date'])
    merged = pd.merge(merged, vol_df, left_on='signal_date', right_on='date', how='left', suffixes=('','_vol'))
    if 'realized_vol_20d' in merged.columns:
        merged['realized_vol_20d'] = pd.to_numeric(merged['realized_vol_20d'], errors='coerce')
        vol_med = merged['realized_vol_20d'].median()
        merged['vol_regime'] = np.where(merged['realized_vol_20d'].isna(),'unknown',
            np.where(merged['realized_vol_20d']>vol_med,'high_vol','low_vol'))
    trend_df = price_df[['date','value']].copy().sort_values('date')
    trend_df['date'] = pd.to_datetime(trend_df['date'])
    trend_df['price_20d_momentum'] = trend_df['value']/trend_df['value'].shift(20)-1
    merged = pd.merge(merged, trend_df[['date','price_20d_momentum']], left_on='signal_date', right_on='date', how='left', suffixes=('','_trend'))
    if 'price_20d_momentum' in merged.columns:
        merged['price_20d_momentum'] = pd.to_numeric(merged['price_20d_momentum'], errors='coerce')
        merged['trend_regime'] = np.where(merged['price_20d_momentum'].isna(),'unknown',
            np.where(merged['price_20d_momentum']>0.02,'uptrend',
            np.where(merged['price_20d_momentum']<-0.02,'downtrend','neutral')))
    return merged

master = {}
for cmd in COMMODITIES:
    master[cmd] = build_master_table(cmd, all_signals[cmd], custom_signals[cmd], price_data[cmd])
    print(f'{cmd}: {len(master[cmd])} rows x {len(master[cmd].columns)} cols, {master[cmd]["signal_date"].min().date()} -> {master[cmd]["signal_date"].max().date()}')
    print(f'  Seasons: {master[cmd]["season"].value_counts().to_dict()}')
    display(master[cmd].tail(2))
master[cmd].describe()

WINTER WHEAT: 452 rows x 27 cols, 2001-11-06 -> 2025-11-25
  Seasons: {'growing': 202, 'harvest': 128, 'planting': 122}


,signal_date,crop_condition_historical_surprise,crop_condition_normalized,crop_condition_normalized_diff,condition_momentum_3w,condition_acceleration,yoy_condition_change,is_extreme_good,is_extreme_bad,ge_zscore,...,fwd_20d_ret,month,year,season,date_vol,realized_vol_20d,vol_regime,date_trend,price_20d_momentum,trend_regime
450,2025-11-18,-0.731982,-0.279759,-0.093511,-0.166667,-1.25,NaN,0.0,0.0,-0.463768,...,-0.086147,11,2025,planting,2025-11-18,0.219683,low_vol,2025-11-18,0.070436,uptrend
451,2025-11-25,0.435870,0.000000,0.279759,0.666667,3.00,NaN,0.0,0.0,-0.133333,...,-0.021152,11,2025,planting,2025-11-25,0.217773,low_vol,2025-11-25,-0.024155,downtrend


,signal_date,crop_condition_historical_surprise,crop_condition_normalized,crop_condition_normalized_diff,condition_momentum_3w,condition_acceleration,yoy_condition_change,is_extreme_good,is_extreme_bad,ge_zscore,...,value,fwd_5d_ret,fwd_10d_ret,fwd_20d_ret,month,year,date_vol,realized_vol_20d,date_trend,price_20d_momentum
count,452,406.000000,452.000000,422.000000,436.000000,436.000000,401.000000,436.000000,436.000000,415.000000,...,452.000000,452.000000,452.000000,452.000000,452.000000,452.000000,452,452.000000,452,452.000000
mean,2013-12-24 13:26:01.061947136,-0.020920,-0.144761,0.040787,0.008983,0.130734,-0.254364,0.211009,0.229358,0.068547,...,557.697456,0.000603,0.003879,0.014063,6.683628,2013.469027,2013-12-24 13:26:01.061947136,0.281628,2013-12-24 13:26:01.061947136,-0.005365
min,2001-11-06 00:00:00,-6.736805,-33.234019,-55.680931,-7.333333,-68.750000,-26.250000,0.000000,0.000000,-3.707627,...,268.750000,-0.140027,-0.184932,-0.243751,3.000000,2001.000000,2001-11-06 00:00:00,0.107395,2001-11-06 00:00:00,-0.275160
25%,2008-03-05 18:00:00,-0.788188,-1.140502,-0.097754,-0.916667,-1.312500,-12.000000,0.000000,0.000000,-0.885063,...,453.812500,-0.032516,-0.041284,-0.047304,5.000000,2007.750000,2008-03-05 18:00:00,0.204938,2008-03-05 18:00:00,-0.048098
50%,2014-01-25 12:00:00,-0.060946,0.118448,0.003948,-0.166667,0.000000,1.250000,0.000000,0.000000,0.093541,...,534.625000,-0.003988,-0.005088,0.003497,6.000000,2013.500000,2014-01-25 12:00:00,0.267129,2014-01-25 12:00:00,-0.007695
75%,2019-12-29 06:00:00,0.792830,0.711129,0.131742,0.416667,1.250000,7.500000,0.000000,0.000000,0.665242,...,661.312500,0.025604,0.036107,0.068823,10.000000,2019.250000,2019-12-29 06:00:00,0.347747,2019-12-29 06:00:00,0.031976
max,2025-11-25 00:00:00,8.923797,60.000000,60.006569,23.666667,153.750000,30.500000,1.000000,1.000000,14.433757,...,1277.500000,0.189741,0.308384,0.535068,11.000000,2025.000000,2025-11-25 00:00:00,0.704326,2025-11-25 00:00:00,0.239308
std,NaN,1.414314,3.667268,4.554409,2.548288,9.524792,13.874847,0.408494,0.420903,1.385714,...,173.295454,0.046827,0.070001,0.100186,2.656940,6.936290,NaN,0.098219,NaN,0.078990


In [7]:
KEY_SIGNALS = ['crop_condition_normalized','crop_condition_normalized_diff',
    'crop_condition_historical_surprise','condition_momentum_3w',
    'ge_zscore','yoy_condition_change','condition_acceleration']

---
## 4. Signal Quality Assessment
IC, ICIR, Hit Rate, Turnover per signal across multiple forward return horizons.

In [8]:
def compute_signal_quality(df, signal_col, return_col='fwd_5d_ret'):
    data = df[[signal_col, return_col, 'year']].dropna().copy()
    if len(data) < 30:
        return {'signal': signal_col, 'n': len(data), 'IC': np.nan, 'ICIR': np.nan, 'hit_rate': np.nan, 'turnover': np.nan}
    ic = data[signal_col].corr(data[return_col])
    yearly_ics = data.groupby('year').apply(lambda g: g[signal_col].corr(g[return_col])).dropna()
    icir = yearly_ics.mean()/yearly_ics.std() if len(yearly_ics)>=3 and yearly_ics.std()>0 else np.nan
    hit = (np.sign(data[signal_col]) == np.sign(data[return_col])).mean()
    signs = np.sign(data[signal_col].sort_index())
    turnover = (signs != signs.shift(1)).sum() / len(data)
    return {'signal': signal_col, 'n': len(data), 'IC': ic, 'ICIR': icir, 'hit_rate': hit, 'turnover': turnover}

def get_signal_columns(df):
    exclude = {'signal_date','date','value','year','month','week_of_year','season',
               'realized_vol_20d','vol_regime','price_20d_momentum','trend_regime','date_price','date_vol'}
    exclude |= {c for c in df.columns if 'fwd_' in c}
    return [c for c in df.columns if c not in exclude and df[c].dtype in ['float64','int64','Int64','float32','int32']]

for cmd in COMMODITIES:
    m = master[cmd]
    sig_cols = get_signal_columns(m)
    print(f'\n{"="*70}\n{COMMODITIES[cmd]["name"]} - Signal Quality\n{"="*70}')
    results = []
    for horizon in FORWARD_HORIZONS:
        for sig in sig_cols:
            r = compute_signal_quality(m, sig, f'fwd_{horizon}d_ret')
            r['horizon'] = f'{horizon}d'
            results.append(r)
    quality_df = pd.DataFrame(results).sort_values('IC', ascending=False, key=abs)
    top = quality_df[quality_df['horizon']=='5d'].head(10)
    print(f'\nTop signals by |IC| (5d fwd):')
    display(top.style.format({'IC':'{:.4f}','ICIR':'{:.2f}','hit_rate':'{:.1%}','turnover':'{:.1%}'}))
    # Bar chart
    fig = go.Figure()
    for horizon in FORWARD_HORIZONS:
        h_df = quality_df[quality_df['horizon']==f'{horizon}d'].sort_values('IC', key=abs)
        fig.add_trace(go.Bar(y=h_df['signal'], x=h_df['IC'], orientation='h', name=f'{horizon}d fwd'))
    fig.update_layout(title=f'{COMMODITIES[cmd]["name"]} - IC by Signal & Horizon',
        barmode='group', height=max(400,len(sig_cols)*25), template='plotly_white', xaxis_title='IC')
    fig.show()
    master[cmd+'_quality'] = quality_df


Wheat - Signal Quality

Top signals by |IC| (5d fwd):


,signal,n,IC,ICIR,hit_rate,turnover,horizon
5,yoy_condition_change,401,0.1351,0.51,54.6%,11.7%,5d
11,condition_index,436,0.1114,0.38,47.2%,0.2%,5d
10,good_excellent_pct,436,0.1073,0.31,47.2%,0.2%,5d
0,crop_condition_historical_surprise,406,0.0962,0.57,49.3%,47.0%,5d
8,ge_zscore,415,0.0856,0.60,54.7%,13.3%,5d
7,is_extreme_bad,436,-0.0848,-0.13,9.9%,4.8%,5d
4,condition_acceleration,436,0.0831,0.21,48.4%,66.7%,5d
9,ge_momentum_3w,436,0.0481,0.47,47.9%,32.3%,5d
2,crop_condition_normalized_diff,422,-0.0439,-0.10,50.7%,49.8%,5d
3,condition_momentum_3w,436,0.0272,0.49,51.4%,29.4%,5d


In [9]:
# Basket monotonicity check
from scipy.stats import spearmanr
for cmd in COMMODITIES:
    m = master[cmd]
    print(f'\n{COMMODITIES[cmd]["name"]} - Basket Analysis:')
    available = [s for s in KEY_SIGNALS if s in m.columns]
    basket_results = []
    for sig in available:
        data = m[[sig,'fwd_5d_ret']].dropna().copy()
        if len(data) < 25: continue
        data['basket'] = pd.qcut(data[sig], q=5, labels=False, duplicates='drop')
        stats = data.groupby('basket')['fwd_5d_ret'].agg(['mean','count']).reset_index()
        mono, _ = spearmanr(stats['basket'], stats['mean'])
        ls = stats['mean'].iloc[-1] - stats['mean'].iloc[0]
        basket_results.append({'signal':sig,'monotonicity':mono,'long_short':ls,'B1':stats['mean'].iloc[0],'B5':stats['mean'].iloc[-1]})
    bdf = pd.DataFrame(basket_results).sort_values('long_short', ascending=False, key=abs)
    display(bdf.style.format({'monotonicity':'{:.3f}','long_short':'{:.4f}','B1':'{:.4f}','B5':'{:.4f}'}))


Wheat - Basket Analysis:


,signal,monotonicity,long_short,B1,B5
5,yoy_condition_change,0.900,0.0164,-0.0071,0.0093
6,condition_acceleration,0.700,0.0140,-0.0083,0.0057
4,ge_zscore,0.900,0.0116,-0.0086,0.0030
2,crop_condition_historical_surprise,0.600,0.0067,-0.0026,0.0041
0,crop_condition_normalized,0.600,0.0065,-0.0041,0.0024
1,crop_condition_normalized_diff,0.600,0.0051,-0.0029,0.0022
3,condition_momentum_3w,0.800,0.0042,-0.0040,0.0002


---
## 5. Seasonal & Regime Analysis
IC broken down by crop season, volatility regime, and price trend regime.

In [10]:
for cmd in COMMODITIES:
    m = master[cmd]
    available = [s for s in KEY_SIGNALS if s in m.columns]
    print(f'\n{"="*70}\n{COMMODITIES[cmd]["name"]} - IC by Season & Regime\n{"="*70}')
    # IC by Season
    season_results = []
    for season in ['planting','growing','harvest','off_season']:
        subset = m[m['season']==season]
        for sig in available:
            data = subset[[sig,'fwd_5d_ret']].dropna()
            if len(data)>=10:
                season_results.append({'season':season,'signal':sig,'IC':data[sig].corr(data['fwd_5d_ret']),'n':len(data)})
    sdf = pd.DataFrame(season_results)
    if len(sdf)>0:
        pivot = sdf.pivot(index='signal',columns='season',values='IC')
        for s in ['planting','growing','harvest','off_season']:
            if s not in pivot.columns: pivot[s]=np.nan
        pivot = pivot[['planting','growing','harvest','off_season']]
        fig = go.Figure(go.Heatmap(z=pivot.values, x=pivot.columns, y=pivot.index,
            colorscale='RdYlGn', zmid=0,
            text=[[f'{v:.3f}' if not pd.isna(v) else '' for v in row] for row in pivot.values],
            texttemplate='%{text}', textfont=dict(size=10)))
        fig.update_layout(title=f'{COMMODITIES[cmd]["name"]} - IC by Signal x Season',
            height=max(400,len(available)*40), template='plotly_white')
        fig.show()
    # IC by Vol & Trend regime
    if 'vol_regime' in m.columns:
        print('\nIC by Vol Regime:')
        vol_res = []
        for regime in ['high_vol','low_vol']:
            for sig in available:
                data = m[m['vol_regime']==regime][[sig,'fwd_5d_ret']].dropna()
                if len(data)>=10: vol_res.append({'vol_regime':regime,'signal':sig,'IC':data[sig].corr(data['fwd_5d_ret'])})
        vdf = pd.DataFrame(vol_res).pivot(index='signal',columns='vol_regime',values='IC')
        display(vdf.style.format('{:.4f}'))
    if 'trend_regime' in m.columns:
        print('\nIC by Trend Regime:')
        trend_res = []
        for regime in ['uptrend','downtrend','neutral']:
            for sig in available:
                data = m[m['trend_regime']==regime][[sig,'fwd_5d_ret']].dropna()
                if len(data)>=10: trend_res.append({'trend_regime':regime,'signal':sig,'IC':data[sig].corr(data['fwd_5d_ret'])})
        tdf = pd.DataFrame(trend_res).pivot(index='signal',columns='trend_regime',values='IC')
        for c in ['uptrend','neutral','downtrend']:
            if c not in tdf.columns: tdf[c]=np.nan
        display(tdf[['uptrend','neutral','downtrend']].style.format('{:.4f}'))


Wheat - IC by Season & Regime



IC by Vol Regime:


vol_regime,high_vol,low_vol
signal,,
condition_acceleration,0.0314,0.1173
condition_momentum_3w,0.0821,-0.0310
crop_condition_historical_surprise,0.0780,0.1223
crop_condition_normalized,0.0277,-0.0317
crop_condition_normalized_diff,0.0613,-0.0768
ge_zscore,0.1084,0.0507
yoy_condition_change,0.1490,0.0958



IC by Trend Regime:


trend_regime,uptrend,neutral,downtrend
signal,,,
condition_acceleration,0.0722,0.0389,0.1134
condition_momentum_3w,0.0189,0.0739,-0.0176
crop_condition_historical_surprise,0.0889,0.0668,0.1265
crop_condition_normalized,-0.0196,-0.0077,0.0056
crop_condition_normalized_diff,-0.0566,-0.0444,-0.0651
ge_zscore,0.1311,0.1228,0.0247
yoy_condition_change,0.1213,0.1952,0.1065


---
## 6. IC Time-Series Heatmap & PnL Simulation
Rolling IC stability and simple long/short backtest for top signals.

In [11]:
for cmd in COMMODITIES:
    m = master[cmd].copy()
    available = [s for s in KEY_SIGNALS if s in m.columns]
    print(f'\n{COMMODITIES[cmd]["name"]} - IC by Year x Signal:')
    ic_year = []
    for yr in sorted(m['year'].unique()):
        for sig in available:
            data = m[m['year']==yr][[sig,'fwd_5d_ret']].dropna()
            if len(data)>=5: ic_year.append({'year':yr,'signal':sig,'IC':data[sig].corr(data['fwd_5d_ret'])})
    icdf = pd.DataFrame(ic_year).pivot(index='signal',columns='year',values='IC')
    fig = go.Figure(go.Heatmap(z=icdf.values, x=[str(y) for y in icdf.columns], y=icdf.index,
        colorscale='RdYlGn', zmid=0, zmin=-0.5, zmax=0.5,
        text=[[f'{v:.2f}' if not pd.isna(v) else '' for v in row] for row in icdf.values],
        texttemplate='%{text}', textfont=dict(size=9)))
    fig.update_layout(title=f'{COMMODITIES[cmd]["name"]} - IC by Year x Signal',
        height=max(400,len(available)*40), template='plotly_white')
    fig.show()


Wheat - IC by Year x Signal:


In [12]:
def simulate_pnl(df, signal_col, return_col='fwd_5d_ret', holding_period=5):
    data = df[['signal_date', signal_col, return_col]].dropna().sort_values('signal_date').copy()
    data['position'] = np.sign(data[signal_col])
    data['strategy_ret'] = data['position'] * data[return_col]
    data['cum_strategy'] = (1 + data['strategy_ret']).cumprod()
    data['cum_buy_hold'] = (1 + data[return_col]).cumprod()
    total = (1 + data['strategy_ret']).prod() - 1
    n_obs = len(data)
    ann_ret = (1+total)**(52/n_obs) - 1 if n_obs>0 else np.nan
    sharpe = data['strategy_ret'].mean()/data['strategy_ret'].std()*np.sqrt(52) if data['strategy_ret'].std()>0 else np.nan
    dd = (data['cum_strategy']/data['cum_strategy'].cummax()-1).min()
    return data, {'ann_ret':ann_ret,'sharpe':sharpe,'max_dd':dd,'n_trades':n_obs}
def simulate_pnl_sum(df, signal_col, return_col="fwd_5d_ret", holding_period=5):
    """Cumulative sum return (non-compounding)."""
    data = df[["signal_date", signal_col, return_col]].dropna().sort_values("signal_date").copy()
    ic = data[signal_col].corr(data[return_col])
    direction = 1 if ic >= 0 else -1
    data["position"] = direction * np.sign(data[signal_col])
    data["strategy_ret"] = data["position"] * data[return_col]
    data["cum_strategy"] = data["strategy_ret"].cumsum()
    data["cum_buy_hold"] = data[return_col].cumsum()
    n_obs = len(data)
    ann_ret = data["strategy_ret"].mean() * 52 if n_obs > 0 else np.nan
    sharpe = data["strategy_ret"].mean()/data["strategy_ret"].std()*np.sqrt(52) if data["strategy_ret"].std()>0 else np.nan
    dd = (data["cum_strategy"] - data["cum_strategy"].cummax()).min()
    return data, {"ann_ret": ann_ret, "sharpe": sharpe, "max_dd": dd, "n_trades": n_obs, "direction": direction, "IC": ic}
    


for cmd in COMMODITIES:
    m = master[cmd]
    print(f'\n{"="*70}\n{COMMODITIES[cmd]["name"]} - PnL Simulation\n{"="*70}')
    available = [s for s in KEY_SIGNALS if s in m.columns]
    pnl_summary = []
    for sig in available:
        result, stats = simulate_pnl_sum(m, sig)
        pnl_summary.append({'signal':sig,**stats})
        fig = go.Figure()
        fig.add_trace(go.Scatter(x=result['signal_date'],y=result['cum_strategy'],name='Strategy'))
        fig.add_trace(go.Scatter(x=result['signal_date'],y=result['cum_buy_hold'],name='Buy&Hold',line=dict(dash='dash')))
        fig.update_layout(title=f'{sig} - Cumulative Return',template='plotly_white',height=300)
        fig.show()
    psdf = pd.DataFrame(pnl_summary).sort_values('sharpe',ascending=False)
    print('\nPnL Summary:')
    display(psdf.style.format({'ann_ret':'{:.2%}','sharpe':'{:.2f}','max_dd':'{:.2%}'}))


Wheat - PnL Simulation



PnL Summary:


,signal,ann_ret,sharpe,max_dd,n_trades,direction,IC
5,yoy_condition_change,34.57%,1.02,-39.96%,401,1,0.135148
3,condition_momentum_3w,27.27%,0.82,-32.40%,436,1,0.027244
4,ge_zscore,27.20%,0.80,-43.71%,415,1,0.085556
2,crop_condition_historical_surprise,17.42%,0.50,-80.87%,406,1,0.096167
6,condition_acceleration,7.77%,0.23,-112.02%,436,1,0.083097
1,crop_condition_normalized_diff,-15.93%,-0.47,-182.43%,422,-1,-0.043889
0,crop_condition_normalized,-19.13%,-0.57,-223.77%,452,-1,-0.004979


In [13]:
print('PnL simulation done.')

PnL simulation done.


---
## 7. Multi-Commodity Summary & Recommendation
Cross-commodity signal ranking and final actionable recommendation.

In [14]:
print('Cross-Commodity Signal Comparison (5d forward):')
cross_summary = []
for cmd in COMMODITIES:
    qdf = master.get(cmd+'_quality')
    if qdf is not None:
        for _, row in qdf[qdf['horizon']=='5d'].iterrows():
            cross_summary.append({'commodity':COMMODITIES[cmd]['name'],'signal':row['signal'],
                'IC':row['IC'],'ICIR':row['ICIR'],'hit_rate':row['hit_rate'],'turnover':row['turnover'],'n':row['n']})
cdf = pd.DataFrame(cross_summary).sort_values('IC',ascending=False,key=abs)
display(cdf.head(20).style.format({'IC':'{:.4f}','ICIR':'{:.2f}','hit_rate':'{:.1%}','turnover':'{:.1%}'}))

# Heatmap
if len(cdf)>0:
    pivot = cdf.pivot_table(index='signal',columns='commodity',values='IC')
    fig = go.Figure(go.Heatmap(z=pivot.values, x=pivot.columns, y=pivot.index,
        colorscale='RdYlGn', zmid=0,
        text=[[f'{v:.3f}' if not pd.isna(v) else '' for v in row] for row in pivot.values],
        texttemplate='%{text}', textfont=dict(size=11)))
    fig.update_layout(title='Cross-Commodity IC Comparison (5d fwd)',template='plotly_white',height=400)
    fig.show()

Cross-Commodity Signal Comparison (5d forward):


,commodity,signal,IC,ICIR,hit_rate,turnover,n
0,Wheat,yoy_condition_change,0.1351,0.51,54.6%,11.7%,401
1,Wheat,condition_index,0.1114,0.38,47.2%,0.2%,436
2,Wheat,good_excellent_pct,0.1073,0.31,47.2%,0.2%,436
3,Wheat,crop_condition_historical_surprise,0.0962,0.57,49.3%,47.0%,406
4,Wheat,ge_zscore,0.0856,0.60,54.7%,13.3%,415
5,Wheat,is_extreme_bad,-0.0848,-0.13,9.9%,4.8%,436
6,Wheat,condition_acceleration,0.0831,0.21,48.4%,66.7%,436
7,Wheat,ge_momentum_3w,0.0481,0.47,47.9%,32.3%,436
8,Wheat,crop_condition_normalized_diff,-0.0439,-0.10,50.7%,49.8%,422
9,Wheat,condition_momentum_3w,0.0272,0.49,51.4%,29.4%,436


In [15]:
print('\n' + '='*70)
print('RECOMMENDATION FRAMEWORK')
print('='*70)
print('''
Signal Selection Criteria:
  1. |IC| > 0.05 and consistent across years (ICIR > 0.5)
  2. Hit rate > 52% (better than random)
  3. Monotonic basket returns (monotonicity > 0.5)
  4. Stronger in growing season (Jun-Aug) when condition data is most informative
  5. Low turnover (< 30%) for implementability

Implementation Notes:
  - Use signal_date (next CBT business day after USDA release) for execution
  - Scale position by signal z-score for risk management
  - Combine top 2-3 uncorrelated signals via equal-weight composite
  - Increase position size during growing season, reduce in off-season
  - Monitor for structural breaks when USDA methodology changes

Next Steps:
  - Validate with out-of-sample data (most recent 2 years)
  - Test with continuous front-month contract for realistic execution
  - Add Soybean planting/harvest progress signals when available
  - Consider cross-commodity signal (corn-soy condition spread)
''')


RECOMMENDATION FRAMEWORK

Signal Selection Criteria:
  1. |IC| > 0.05 and consistent across years (ICIR > 0.5)
  2. Hit rate > 52% (better than random)
  3. Monotonic basket returns (monotonicity > 0.5)
  4. Stronger in growing season (Jun-Aug) when condition data is most informative
  5. Low turnover (< 30%) for implementability

Implementation Notes:
  - Use signal_date (next CBT business day after USDA release) for execution
  - Scale position by signal z-score for risk management
  - Combine top 2-3 uncorrelated signals via equal-weight composite
  - Increase position size during growing season, reduce in off-season
  - Monitor for structural breaks when USDA methodology changes

Next Steps:
  - Validate with out-of-sample data (most recent 2 years)
  - Test with continuous front-month contract for realistic execution
  - Add Soybean planting/harvest progress signals when available
  - Consider cross-commodity signal (corn-soy condition spread)



In [16]:
print('\n' + '='*70)
print('CURRENT SIGNAL SNAPSHOT')
print('='*70)
for cmd in COMMODITIES:
    print(f'\n{COMMODITIES[cmd]["name"]}:')
    cs = custom_signals[cmd]
    latest = cs.sort_values('signal_date').iloc[-1]
    print(f'  Latest signal date: {latest["signal_date"].date()}')
    print(f'  Good+Excellent: {latest["good_excellent_pct"]:.1f}%')
    print(f'  Condition Index: {latest["condition_index"]:.2f}')
    print(f'  GE Z-Score: {latest["ge_zscore"]:.2f}' if not pd.isna(latest['ge_zscore']) else '  GE Z-Score: N/A')
    print(f'  YoY Change: {latest["yoy_condition_change"]:.2f}' if not pd.isna(latest['yoy_condition_change']) else '  YoY Change: N/A')
    print(f'  3w Momentum: {latest["condition_momentum_3w"]:.3f}' if not pd.isna(latest['condition_momentum_3w']) else '  3w Momentum: N/A')
    if latest['is_extreme_good']: print('  *** EXTREME GOOD condition ***')
    if latest['is_extreme_bad']: print('  *** EXTREME BAD condition ***')


CURRENT SIGNAL SNAPSHOT

Wheat:
  Latest signal date: 2026-04-07
  Good+Excellent: 35.0%
  Condition Index: 71.50
  GE Z-Score: -0.87
  YoY Change: N/A
  3w Momentum: -3.250


---
## 8. Additional Signals & Feature Engineering

In [17]:
def build_extended_signals(master_dict):
    for cmd in master_dict:
        if cmd in [ 'CORN_quality', 'SOYBEANS_quality', 'WINTER WHEAT_quality']:
            continue
        m = master_dict[cmd].copy().sort_values('signal_date').reset_index(drop=True)
        if 'crop_condition_historical_surprise' in m.columns and 'condition_momentum_3w' in m.columns:
            m['surprise_x_momentum'] = m['crop_condition_historical_surprise'] * m['condition_momentum_3w']
        if 'ge_zscore' in m.columns and 'condition_acceleration' in m.columns:
            m['zscore_x_accel'] = m['ge_zscore'] * m['condition_acceleration']
        if 'crop_condition_normalized' in m.columns and 'condition_momentum_3w' in m.columns:
            m['level_x_momentum'] = m['crop_condition_normalized'] * m['condition_momentum_3w']
        if 'yoy_condition_change' in m.columns and 'ge_zscore' in m.columns:
            m['yoy_x_zscore'] = m['yoy_condition_change'] * m['ge_zscore']
        for col in [c for c in m.columns if c in ['crop_condition_historical_surprise','condition_momentum_3w','ge_zscore']]:
            m[f'{col}_lag1'] = m[col].shift(1)
            m[f'{col}_lag2'] = m[col].shift(2)
        if 'crop_condition_historical_surprise' in m.columns:
            m['surprise_diff'] = m['crop_condition_historical_surprise'].diff()
        if 'realized_vol_20d' in m.columns and 'crop_condition_normalized_diff' in m.columns:
            m['vol_adj_diff'] = m['crop_condition_normalized_diff'] / m['realized_vol_20d'].replace(0, np.nan)
        if 'season' in m.columns and 'ge_zscore' in m.columns:
            for season in ['growing','harvest','planting']:
                m[f'ge_zscore_{season}'] = m['ge_zscore'] * (m['season'] == season).astype(int)
        if 'is_extreme_bad' in m.columns and 'condition_momentum_3w' in m.columns:
            m['extreme_bad_deteriorating'] = m['is_extreme_bad'] * (m['condition_momentum_3w'] < 0).astype(int)
        if 'is_extreme_good' in m.columns and 'condition_momentum_3w' in m.columns:
            m['extreme_good_improving'] = m['is_extreme_good'] * (m['condition_momentum_3w'] > 0).astype(int)
        master_dict[cmd] = m
        print(f'{cmd}: added extended signals -> {m.shape[1]} columns')
    return master_dict

master = build_extended_signals(master)

WINTER WHEAT: added extended signals -> 44 columns


In [18]:
print("Module 8 done.")

Module 8 done.


---
## 9. Signal Correlation Analysis
Understand inter-signal correlations to identify redundant and complementary signals.

In [19]:
def get_all_signal_cols(df):
    exclude = {"signal_date","date","value","year","month","week_of_year","season",
               "realized_vol_20d","vol_regime","price_20d_momentum","trend_regime",
               "date_price","date_vol","good_excellent_pct","condition_index"}
    exclude |= {c for c in df.columns if "fwd_" in c}
    return [c for c in df.columns if c not in exclude and df[c].dtype in ["float64","int64","Int64","float32","int32"]]

for cmd in COMMODITIES:
    m = master[cmd]
    sig_cols = get_all_signal_cols(m)
    corr_matrix = m[sig_cols].corr()
    fig = go.Figure(go.Heatmap(z=corr_matrix.values, x=corr_matrix.columns, y=corr_matrix.index,
        colorscale="RdBu_r", zmid=0, zmin=-1, zmax=1,
        text=[[f"{v:.2f}" for v in row] for row in corr_matrix.values],
        texttemplate="%{text}", textfont=dict(size=7)))
    fig.update_layout(title=f"{cmd} - Signal Correlation", width=900, height=800, template="plotly_white")
    fig.show()
    print(f"Highly correlated pairs (|corr| > 0.6):")
    for ii in range(len(corr_matrix)):
        for jj in range(ii+1, len(corr_matrix)):
            val = corr_matrix.iloc[ii,jj]
            if abs(val) > 0.6:
                print(f"  {corr_matrix.index[ii]:40s} <-> {corr_matrix.columns[jj]:40s}: {val:.3f}")
    print(f"\nLow correlation pairs (|corr| < 0.3) with highest avg |IC|:")
    low_pairs = []
    for ii in range(len(corr_matrix)):
        for jj in range(ii+1, len(corr_matrix)):
            val = corr_matrix.iloc[ii,jj]
            if abs(val) < 0.3:
                s1, s2 = corr_matrix.index[ii], corr_matrix.columns[jj]
                ic1 = m[s1].corr(m["fwd_5d_ret"]) if s1 in m.columns else 0
                ic2 = m[s2].corr(m["fwd_5d_ret"]) if s2 in m.columns else 0
                low_pairs.append({"sig1":s1,"sig2":s2,"corr":val,"ic1":ic1,"ic2":ic2,"avg_ic":(abs(ic1)+abs(ic2))/2})
    if low_pairs:
        ldf = pd.DataFrame(low_pairs).sort_values("avg_ic",ascending=False).head(10)
        display(ldf.style.format({"corr":"{:.3f}","ic1":"{:.4f}","ic2":"{:.4f}","avg_ic":"{:.4f}"}))

Highly correlated pairs (|corr| > 0.6):
  crop_condition_historical_surprise       <-> surprise_diff                           : 0.680
  crop_condition_normalized                <-> crop_condition_normalized_diff          : 0.609
  crop_condition_normalized                <-> ge_zscore                               : 0.781
  crop_condition_normalized                <-> zscore_x_accel                          : 0.747
  crop_condition_normalized                <-> level_x_momentum                        : 0.727
  crop_condition_normalized                <-> ge_zscore_planting                      : 0.820
  crop_condition_normalized_diff           <-> zscore_x_accel                          : 0.878
  crop_condition_normalized_diff           <-> vol_adj_diff                            : 0.995
  condition_momentum_3w                    <-> ge_momentum_3w                          : 0.928
  is_extreme_good                          <-> extreme_good_improving                  : 0.742
  is_extre

,sig1,sig2,corr,ic1,ic2,avg_ic
4,crop_condition_historical_surprise,yoy_condition_change,-0.037,0.0962,0.1351,0.1157
105,yoy_condition_change,surprise_diff,-0.026,0.1351,0.0938,0.1145
21,crop_condition_historical_surprise,ge_zscore_harvest,0.032,0.0962,0.1281,0.1121
250,surprise_diff,ge_zscore_harvest,0.061,0.0938,0.1281,0.1109
76,condition_acceleration,yoy_condition_change,-0.059,0.0831,0.1351,0.1091
92,condition_acceleration,ge_zscore_harvest,-0.002,0.0831,0.1281,0.1056
262,ge_zscore_harvest,extreme_bad_deteriorating,-0.218,0.1281,-0.0669,0.0975
258,ge_zscore_growing,ge_zscore_harvest,-0.001,0.0612,0.1281,0.0947
98,yoy_condition_change,zscore_x_accel,0.042,0.1351,-0.0530,0.0941
104,yoy_condition_change,condition_momentum_3w_lag2,0.173,0.1351,0.0510,0.0931


---
## 10. Composite Signal Building
Build optimized composite signals: equal-weight, IC-weighted, and rolling-IC adaptive.

In [20]:
for cmd in COMMODITIES:
    m = master[cmd].copy().sort_values("signal_date").reset_index(drop=True)
    sig_cols = get_all_signal_cols(m)
    # Select uncorrelated signals with decent IC
    sig_ics = {}
    for s in sig_cols:
        valid = m[s].notna() & m["fwd_5d_ret"].notna()
        if valid.sum() > 30:
            sig_ics[s] = abs(m.loc[valid, s].corr(m.loc[valid, "fwd_5d_ret"]))
    candidates = sorted(sig_ics, key=sig_ics.get, reverse=True)[:10]
    # Remove highly correlated duplicates
    corr = m[candidates].corr().abs()
    selected = []
    for s in candidates:
        if not any(abs(corr.loc[s, x]) > 0.7 for x in selected if x in corr.columns):
            selected.append(s)
        if len(selected) >= 6: break
    print(f"\n{cmd} - Selected {len(selected)} uncorrelated signals:")
    for s in selected:
        ic_val = m[s].corr(m["fwd_5d_ret"])
        print(f"  {s:45s} IC={ic_val:.4f}")
    # Z-score standardize
    for s in selected:
        m[f"{s}_z"] = (m[s] - m[s].mean()) / m[s].std()
    z_cols = [f"{s}_z" for s in selected]
    # Method 1: Equal Weight
    m["composite_equal"] = m[z_cols].mean(axis=1)
    # Method 2: IC-Weighted
    ic_w = {}
    for s in selected:
        ic = m[s].corr(m["fwd_5d_ret"])
        ic_w[s] = ic if not pd.isna(ic) else 0
    total_ic = sum(abs(v) for v in ic_w.values())
    if total_ic > 0:
        ic_w = {k: abs(v)/total_ic * np.sign(v) for k,v in ic_w.items()}
    m["composite_ic_weighted"] = sum(m[f"{s}_z"] * w for s,w in ic_w.items())
    # Method 3: Rolling IC-Weighted (adaptive)
    for s in selected:
        m[f"{s}_ric"] = m[s].rolling(22, min_periods=20).corr(m["fwd_5d_ret"])
    ric_vals = []
    for idx in range(len(m)):
        rics = {s: m.iloc[idx][f"{s}_ric"] for s in selected}
        rics = {k: v for k,v in rics.items() if not pd.isna(v)}
        tot = sum(abs(v) for v in rics.values())
        if tot > 0:
            ws = {k: abs(v)/tot * np.sign(v) for k,v in rics.items()}
            val = sum(m.iloc[idx][f"{s}_z"] * w for s,w in ws.items() if f"{s}_z" in m.columns)
        else:
            val = np.nan
        ric_vals.append(val)
    m["composite_rolling_ic"] = ric_vals
    # Evaluate composites
    comp_names = ["composite_equal","composite_ic_weighted","composite_rolling_ic"]
    print(f"\nComposite Signal Quality:")
    comp_results = []
    for cn in comp_names:
        for h in FORWARD_HORIZONS:
            r = compute_signal_quality(m, cn, f"fwd_{h}d_ret")
            r["horizon"] = f"{h}d"
            comp_results.append(r)
    cdf = pd.DataFrame(comp_results).sort_values("IC", ascending=False, key=abs)
    display(cdf.style.format({"IC":"{:.4f}","ICIR":"{:.2f}","hit_rate":"{:.1%}","turnover":"{:.1%}"}))
    # PnL for best composite
    best = cdf.iloc[0]["signal"]
    result, stats = simulate_pnl_sum(m, best)
    print(f"Best: {best} -> Sharpe={stats['sharpe']}, AnnRet={stats['ann_ret']}, MaxDD={stats['max_dd']:.2%}")
    fig = go.Figure()
    fig.add_trace(go.Scatter(x=result["signal_date"], y=result["cum_strategy"], name="Composite"))
    fig.add_trace(go.Scatter(x=result["signal_date"], y=result["cum_buy_hold"], name="Buy&Hold", line=dict(dash="dash")))
    fig.update_layout(title=f"{cmd} {best} - Cumulative Return", template="plotly_white", height=800)
    fig.show()
    master[cmd] = m


WINTER WHEAT - Selected 6 uncorrelated signals:
  yoy_condition_change                          IC=0.1351
  ge_zscore_harvest                             IC=0.1281
  crop_condition_historical_surprise            IC=0.0962
  surprise_diff                                 IC=0.0938
  ge_zscore                                     IC=0.0856
  is_extreme_bad                                IC=-0.0848

Composite Signal Quality:


,signal,n,IC,ICIR,hit_rate,turnover,horizon
6,composite_rolling_ic,394,0.3483,1.41,61.2%,36.5%,5d
5,composite_ic_weighted,336,0.2620,0.21,53.0%,31.5%,20d
7,composite_rolling_ic,394,0.2531,0.64,57.6%,36.5%,10d
8,composite_rolling_ic,394,0.2391,0.35,54.8%,36.5%,20d
4,composite_ic_weighted,336,0.2224,0.22,55.4%,31.5%,10d
3,composite_ic_weighted,336,0.2157,0.31,57.7%,31.5%,5d
2,composite_equal,449,0.1166,0.13,53.0%,38.3%,20d
0,composite_equal,449,0.1064,0.55,56.8%,38.3%,5d
1,composite_equal,449,0.0887,0.49,53.2%,38.3%,10d


Best: composite_rolling_ic -> Sharpe=2.259811586774295, AnnRet=0.738762893254648, MaxDD=-28.44%


---
## 11. Grid Search - Best Signal Combinations
Try all combinations of 2-4 signals to find the best composite.

In [21]:
from itertools import combinations

for cmd in COMMODITIES:
    m = master[cmd]
    base_sigs = [s for s in KEY_SIGNALS if s in m.columns]
    extra = ["surprise_x_momentum", "zscore_x_accel", "level_x_momentum",
             "yoy_x_zscore", "vol_adj_diff"]
    extra = [s for s in extra if s in m.columns]
    candidates = base_sigs + extra
    candidates = [s for s in candidates if m[s].notna().sum() > 50]
    # Standardize
    for s in candidates:
        m[f"{s}_z"] = (m[s] - m[s].mean()) / m[s].std()
    print(f"\n{cmd} - Grid Search over {len(candidates)} candidates")
    best_results = []
    for n_combo in [2, 3, 4]:
        for combo in combinations(candidates, n_combo):
            z_cols = [f"{s}_z" for s in combo]
            composite = m[z_cols].mean(axis=1)
            valid = composite.notna() & m["fwd_5d_ret"].notna()
            if valid.sum() < 50: continue
            ic5 = composite[valid].corr(m.loc[valid, "fwd_5d_ret"])
            ic10 = composite[valid].corr(m.loc[valid, "fwd_10d_ret"]) if "fwd_10d_ret" in m.columns else np.nan
            hit = (np.sign(composite[valid]) == np.sign(m.loc[valid, "fwd_5d_ret"])).mean()
            best_results.append({"signals": " + ".join(combo), "n": n_combo,
                "IC_5d": ic5, "IC_10d": ic10, "hit_rate": hit, "n_obs": valid.sum()})
    gdf = pd.DataFrame(best_results).sort_values("IC_5d", ascending=False, key=abs)
    print(f"\nTop 10 combinations by |IC| (5d):")
    display(gdf.head(10).style.format({"IC_5d":"{:.4f}","IC_10d":"{:.4f}","hit_rate":"{:.1%}"}))
    # PnL for top combo
    top_sigs = gdf.iloc[0]["signals"].split(" + ")
    top_z = [f"{s}_z" for s in top_sigs if f"{s}_z" in m.columns]
    m["best_grid_composite"] = m[top_z].mean(axis=1)
    result, stats = simulate_pnl_sum(m, "best_grid_composite")
    print(f"Best Grid Combo: Sharpe={stats['sharpe']:.2f}, AnnRet={stats['ann_ret']:.2%}")
    fig = go.Figure()
    fig.add_trace(go.Scatter(x=result["signal_date"], y=result["cum_strategy"], name="Grid Strategy"))
    fig.add_trace(go.Scatter(x=result["signal_date"], y=result["cum_buy_hold"], name="Buy&Hold", line=dict(dash="dash")))
    fig.update_layout(title=f"{cmd} Best Grid Composite - Cumulative Return", template="plotly_white", height=400)
    fig.show()
    master[cmd] = m


WINTER WHEAT - Grid Search over 12 candidates

Top 10 combinations by |IC| (5d):


,signals,n,IC_5d,IC_10d,hit_rate,n_obs
45,yoy_condition_change + condition_acceleration,2,0.1530,0.1242,55.7%,436
230,ge_zscore + yoy_condition_change + condition_acceleration,3,0.1466,0.1424,57.1%,436
711,ge_zscore + yoy_condition_change + condition_acceleration + surprise_x_momentum,4,0.1401,0.1029,58.0%,436
251,yoy_condition_change + condition_acceleration + surprise_x_momentum,3,0.1388,0.0723,56.2%,436
676,condition_momentum_3w + yoy_condition_change + condition_acceleration + surprise_x_momentum,4,0.1370,0.0740,55.7%,436
209,condition_momentum_3w + yoy_condition_change + condition_acceleration,3,0.1354,0.1060,54.8%,436
655,condition_momentum_3w + ge_zscore + yoy_condition_change + condition_acceleration,4,0.1345,0.1268,53.4%,436
181,crop_condition_historical_surprise + yoy_condition_change + condition_acceleration,3,0.1330,0.0787,54.1%,449
23,crop_condition_historical_surprise + yoy_condition_change,2,0.1238,0.0910,54.2%,443
39,ge_zscore + condition_acceleration,2,0.1197,0.1015,57.6%,436


Best Grid Combo: Sharpe=1.01, AnnRet=34.01%


---
## 12. Walk-Forward Lasso Regression (Out-of-Sample)
Rolling regression with Lasso for automatic feature selection and OOS validation.

In [22]:
from sklearn.linear_model import LassoCV
from sklearn.preprocessing import StandardScaler

for cmd in COMMODITIES:
    m = master[cmd].copy().sort_values("signal_date").reset_index(drop=True)
    features = [s for s in KEY_SIGNALS if s in m.columns]
    extra = ["surprise_x_momentum", "zscore_x_accel", "vol_adj_diff"]
    features += [s for s in extra if s in m.columns]
    print(f"\n{cmd} - Walk-Forward Lasso, features: {features}")
    data = m[["signal_date","year"] + features + ["fwd_5d_ret"]].dropna().copy()
    if len(data) < 100:
        print("Not enough data"); continue
    all_years = sorted(data["year"].unique())
    oos_preds = []
    for yr in all_years:
        yr_idx = all_years.index(yr)
        if yr_idx < 8: continue
        train = data[data["year"] < yr]
        test = data[data["year"] == yr]
        if len(train) < 50 or len(test) < 5: continue
        scaler = StandardScaler()
        X_train = scaler.fit_transform(train[features])
        y_train = train["fwd_5d_ret"].values
        X_test = scaler.transform(test[features])
        try:
            model = LassoCV(cv=min(5, len(train)//20), max_iter=5000, random_state=42)
            model.fit(X_train, y_train)
            preds = model.predict(X_test)
            coefs = {f: c for f, c in zip(features, model.coef_) if abs(c) > 1e-6}
            if coefs: print(f"  {yr}: non-zero coefs = {coefs}")
            for j, idx in enumerate(test.index):
                oos_preds.append({"idx": idx, "pred": preds[j], "actual": test.loc[idx, "fwd_5d_ret"],
                                  "signal_date": test.loc[idx, "signal_date"]})
        except Exception as e:
            print(f"  {yr}: failed - {e}")
    if not oos_preds: print("No OOS predictions"); continue
    oos_df = pd.DataFrame(oos_preds).set_index("idx")
    m["ml_prediction"] = np.nan
    m.loc[oos_df.index, "ml_prediction"] = oos_df["pred"]
    valid = m["ml_prediction"].notna() & m["fwd_5d_ret"].notna()
    if valid.sum() > 20:
        oos_ic = m.loc[valid, "ml_prediction"].corr(m.loc[valid, "fwd_5d_ret"])
        oos_hit = (np.sign(m.loc[valid, "ml_prediction"]) == np.sign(m.loc[valid, "fwd_5d_ret"])).mean()
        print(f"\nOOS Results: IC={oos_ic:.4f}, Hit={oos_hit:.1%}, N={valid.sum()}")
        result, stats = simulate_pnl_sum(m[valid].copy().reset_index(drop=True), "ml_prediction")
        print(f"  Sharpe={stats['sharpe']:.2f}, AnnRet={stats['ann_ret']:.2%}, MaxDD={stats['max_dd']:.2%}")
        fig = go.Figure()
        fig.add_trace(go.Scatter(x=result["signal_date"], y=result["cum_strategy"], name="Lasso OOS"))
        fig.add_trace(go.Scatter(x=result["signal_date"], y=result["cum_buy_hold"], name="Buy&Hold", line=dict(dash="dash")))
        fig.update_layout(title=f"{cmd} Walk-Forward Lasso (OOS)", template="plotly_white", height=400)
        fig.show()
    master[cmd] = m


WINTER WHEAT - Walk-Forward Lasso, features: ['crop_condition_normalized', 'crop_condition_normalized_diff', 'crop_condition_historical_surprise', 'condition_momentum_3w', 'ge_zscore', 'yoy_condition_change', 'condition_acceleration', 'surprise_x_momentum', 'zscore_x_accel', 'vol_adj_diff']
  2012: non-zero coefs = {'yoy_condition_change': np.float64(0.0003672880160053923)}
  2013: non-zero coefs = {'yoy_condition_change': np.float64(0.002398152697985032)}
  2014: non-zero coefs = {'yoy_condition_change': np.float64(0.004518983060360997), 'condition_acceleration': np.float64(0.0015097996483142526)}
  2015: non-zero coefs = {'crop_condition_historical_surprise': np.float64(0.0004955200161543325), 'yoy_condition_change': np.float64(0.004941141997204139), 'condition_acceleration': np.float64(0.0009156676027182224)}
  2016: non-zero coefs = {'yoy_condition_change': np.float64(0.00225943295507905)}
  2017: non-zero coefs = {'yoy_condition_change': np.float64(0.0031198213892242644)}
  2018:

---
## 13. In-Sample vs Out-of-Sample Validation
Split data into IS (before 2023) and OOS (2023+) to check signal robustness.

In [23]:
OOS_START_YEAR = 2023

for cmd in COMMODITIES:
    m = master[cmd].copy()
    is_data = m[m["year"] < OOS_START_YEAR]
    oos_data = m[m["year"] >= OOS_START_YEAR]
    sig_cols = get_all_signal_cols(m) if "get_all_signal_cols" in dir() else KEY_SIGNALS
    sig_cols = [s for s in sig_cols if s in m.columns]
    print(f"\n{cmd} - IS: {len(is_data)} rows, OOS: {len(oos_data)} rows")
    results = []
    for sig in sig_cols:
        for label, data in [("IS", is_data), ("OOS", oos_data)]:
            valid = data[[sig, "fwd_5d_ret"]].dropna()
            if len(valid) < 10: continue
            ic = valid[sig].corr(valid["fwd_5d_ret"])
            hit = (np.sign(valid[sig]) == np.sign(valid["fwd_5d_ret"])).mean()
            results.append({"signal": sig, "period": label, "IC": ic, "hit_rate": hit, "n": len(valid)})
    rdf = pd.DataFrame(results)
    if len(rdf) == 0: continue
    pivot_ic = rdf.pivot(index="signal", columns="period", values="IC")
    pivot_hit = rdf.pivot(index="signal", columns="period", values="hit_rate")
    summary = pd.DataFrame({
        "IC_IS": pivot_ic.get("IS", pd.Series(dtype=float)),
        "IC_OOS": pivot_ic.get("OOS", pd.Series(dtype=float)),
        "IC_decay": pivot_ic.get("OOS", pd.Series(dtype=float)) - pivot_ic.get("IS", pd.Series(dtype=float)),
        "Hit_IS": pivot_hit.get("IS", pd.Series(dtype=float)),
        "Hit_OOS": pivot_hit.get("OOS", pd.Series(dtype=float)),
    }).sort_values("IC_IS", ascending=False, key=abs)
    print(f"  Top signals (IS vs OOS):")
    display(summary.head(15).style.format({"IC_IS":"{:.4f}","IC_OOS":"{:.4f}","IC_decay":"{:.4f}","Hit_IS":"{:.1%}","Hit_OOS":"{:.1%}"}))
    # PnL comparison for top signal
    top_sig = summary.index[0]
    if len(oos_data[[top_sig, "fwd_5d_ret"]].dropna()) > 5:
        result_oos, stats_oos = simulate_pnl_sum(oos_data, top_sig)
        print(f"  OOS PnL ({top_sig}): Sharpe={stats_oos['sharpe']:.2f}, AnnRet={stats_oos['ann_ret']:.2%}, MaxDD={stats_oos['max_dd']:.2%}")
        fig = go.Figure()
        fig.add_trace(go.Scatter(x=result_oos["signal_date"], y=result_oos["cum_strategy"], name="OOS Strategy"))
        fig.add_trace(go.Scatter(x=result_oos["signal_date"], y=result_oos["cum_buy_hold"], name="Buy&Hold", line=dict(dash="dash")))
        fig.update_layout(title=f"{cmd} OOS PnL ({top_sig})", template="plotly_white", height=350)
        fig.show()



WINTER WHEAT - IS: 397 rows, OOS: 55 rows
  Top signals (IS vs OOS):


,IC_IS,IC_OOS,IC_decay,Hit_IS,Hit_OOS
signal,,,,,
composite_rolling_ic,0.3214,0.5439,0.2226,60.2%,68.0%
composite_ic_weighted,0.1906,0.5010,0.3104,57.4%,61.3%
best_grid_composite,0.1504,0.1964,0.0460,55.8%,55.6%
ge_zscore_harvest_z,0.1427,-0.0922,-0.2348,54.6%,50.0%
ge_zscore_harvest,0.1427,-0.0922,-0.2348,17.7%,16.7%
yoy_condition_change,0.1351,0.1254,-0.0097,54.9%,51.4%
yoy_condition_change_z,0.1351,0.1254,-0.0097,55.5%,51.4%
is_extreme_bad_z,-0.0881,-0.0711,0.0170,48.2%,53.7%
is_extreme_bad,-0.0881,-0.0711,0.0170,9.9%,9.3%


  OOS PnL (composite_rolling_ic): Sharpe=3.46, AnnRet=108.83%, MaxDD=-11.69%


---
## 14. Permutation Test + Multiple Testing Correction
Empirical significance via shuffled signal dates and Benjamini-Hochberg correction.

In [24]:
from scipy.stats import spearmanr

N_PERMUTATIONS = 500

for cmd in COMMODITIES:
    m = master[cmd].copy()
    sig_cols = get_all_signal_cols(m) if "get_all_signal_cols" in dir() else KEY_SIGNALS
    sig_cols = [s for s in sig_cols if s in m.columns]
    print(f"\n{cmd} - Permutation Test ({N_PERMUTATIONS} shuffles)")
    perm_results = []
    for sig in sig_cols:
        valid = m[[sig, "fwd_5d_ret"]].dropna()
        if len(valid) < 30: continue
        actual_ic = valid[sig].corr(valid["fwd_5d_ret"])
        # Permutation: shuffle signal values
        null_ics = []
        for _ in range(N_PERMUTATIONS):
            shuffled = valid[sig].sample(frac=1, replace=False).reset_index(drop=True)
            shuffled.index = valid.index
            null_ic = shuffled.corr(valid["fwd_5d_ret"])
            null_ics.append(null_ic)
        null_ics = np.array(null_ics)
        # p-value: fraction of null |IC| >= |actual IC|
        p_value = (np.abs(null_ics) >= np.abs(actual_ic)).mean()
        perm_results.append({"signal": sig, "actual_IC": actual_ic,
            "null_IC_mean": null_ics.mean(), "null_IC_std": null_ics.std(),
            "p_value": p_value, "n": len(valid)})
    if not perm_results: continue
    prdf = pd.DataFrame(perm_results).sort_values("p_value")
    # Benjamini-Hochberg correction
    m_tests = len(prdf)
    prdf = prdf.sort_values("p_value").reset_index(drop=True)
    prdf["bh_rank"] = range(1, len(prdf) + 1)
    prdf["bh_threshold"] = prdf["bh_rank"] / m_tests * 0.10  # 10% FDR
    prdf["significant_10pct_fdr"] = prdf["p_value"] <= prdf["bh_threshold"]
    print(f"\n  Tested {m_tests} signals. Significant at 10% FDR:")
    sig_df = prdf[prdf["significant_10pct_fdr"]]
    display(prdf.style.format({"actual_IC":"{:.4f}","null_IC_mean":"{:.4f}","null_IC_std":"{:.4f}","p_value":"{:.4f}","bh_threshold":"{:.4f}"}))
    if len(sig_df) > 0:
        print(f"  -> {len(sig_df)} signals survive BH correction at 10% FDR:")
        for _, r in sig_df.iterrows():
            print(f"     {r['signal']:45s} IC={r['actual_IC']:.4f}  p={r['p_value']:.4f}")
    else:
        print("  -> No signals survive BH correction. Consider larger sample or fewer tests.")
    # Histogram of null distribution for top signal
    top = prdf.iloc[0]
    top_null = []
    valid = m[[top["signal"], "fwd_5d_ret"]].dropna()
    for _ in range(N_PERMUTATIONS):
        shuffled = valid[top["signal"]].sample(frac=1, replace=False).reset_index(drop=True)
        shuffled.index = valid.index
        top_null.append(shuffled.corr(valid["fwd_5d_ret"]))
    fig = go.Figure()
    fig.add_trace(go.Histogram(x=top_null, name="Null IC", nbinsx=50))
    fig.add_vline(x=top["actual_IC"], line_dash="dash", line_color="red", annotation_text=f"Actual IC={top['actual_IC']:.4f}")
    fig.update_layout(title=f"{cmd} - Null IC Distribution ({top['signal']})", template="plotly_white", height=350)
    fig.show()



WINTER WHEAT - Permutation Test (500 shuffles)

  Tested 53 signals. Significant at 10% FDR:


,signal,actual_IC,null_IC_mean,null_IC_std,p_value,n,bh_rank,bh_threshold,significant_10pct_fdr
0,composite_rolling_ic,0.3483,0.0034,0.0524,0.0000,394,1,0.0019,True
1,composite_ic_weighted,0.2157,-0.0006,0.0539,0.0000,336,2,0.0038,True
2,best_grid_composite,0.1530,0.0009,0.0460,0.0040,436,3,0.0057,True
3,yoy_condition_change_z,0.1351,0.0048,0.0512,0.0060,401,4,0.0075,True
4,ge_zscore_harvest,0.1281,-0.0019,0.0479,0.0060,415,5,0.0094,True
5,yoy_condition_change,0.1351,0.0021,0.0502,0.0080,401,6,0.0113,True
6,ge_zscore_harvest_z,0.1281,-0.0008,0.0486,0.0120,415,7,0.0132,True
7,composite_equal,0.1064,-0.0006,0.0467,0.0140,449,8,0.0151,True
8,crop_condition_historical_surprise,0.0962,-0.0005,0.0474,0.0400,406,9,0.0170,False
9,ml_prediction,0.1362,0.0011,0.0602,0.0400,247,10,0.0189,False


  -> 8 signals survive BH correction at 10% FDR:
     composite_rolling_ic                          IC=0.3483  p=0.0000
     composite_ic_weighted                         IC=0.2157  p=0.0000
     best_grid_composite                           IC=0.1530  p=0.0040
     yoy_condition_change_z                        IC=0.1351  p=0.0060
     ge_zscore_harvest                             IC=0.1281  p=0.0060
     yoy_condition_change                          IC=0.1351  p=0.0080
     ge_zscore_harvest_z                           IC=0.1281  p=0.0120
     composite_equal                               IC=0.1064  p=0.0140


##### Debug

In [25]:
a = master['CORN']
a[a['signal_date']>'2023-01-01']['fwd_5d_ret'].head()

KeyError: 'CORN'

In [ ]:
a.columns

Index(['signal_date', 'crop_condition_historical_surprise',
       'crop_condition_normalized', 'crop_condition_normalized_diff',
       'condition_momentum_3w', 'condition_acceleration',
       'yoy_condition_change', 'is_extreme_good', 'is_extreme_bad',
       'ge_zscore', 'ge_momentum_3w', 'good_excellent_pct', 'condition_index',
       'date', 'value', 'fwd_5d_ret', 'fwd_10d_ret', 'fwd_20d_ret', 'month',
       'year', 'season', 'date_vol', 'realized_vol_20d', 'vol_regime',
       'date_trend', 'price_20d_momentum', 'trend_regime',
       'surprise_x_momentum', 'zscore_x_accel', 'level_x_momentum',
       'yoy_x_zscore', 'crop_condition_historical_surprise_lag1',
       'crop_condition_historical_surprise_lag2', 'condition_momentum_3w_lag1',
       'condition_momentum_3w_lag2', 'ge_zscore_lag1', 'ge_zscore_lag2',
       'surprise_diff', 'vol_adj_diff', 'ge_zscore_growing',
       'ge_zscore_harvest', 'ge_zscore_planting', 'extreme_bad_deteriorating',
       'extreme_good_improving'

In [ ]:
master['WINTER WHEAT'].tail()

,signal_date,crop_condition_historical_surprise,crop_condition_normalized,crop_condition_normalized_diff,condition_momentum_3w,condition_acceleration,yoy_condition_change,is_extreme_good,is_extreme_bad,ge_zscore,...,crop_condition_normalized_z,crop_condition_normalized_diff_z,crop_condition_historical_surprise_z,condition_momentum_3w_z,surprise_x_momentum_z,level_x_momentum_z,yoy_x_zscore_z,vol_adj_diff_z,best_grid_composite,ml_prediction
447,2025-06-24,-1.516231,0.121448,-0.154969,-0.500000,1.00,NaN,0.0,0.0,0.104358,...,0.072590,-0.042982,-1.057270,-0.199735,0.136915,-0.087638,NaN,-0.037404,0.091264,NaN
448,2025-07-01,-0.683619,0.115449,-0.005999,-1.083333,0.75,NaN,0.0,0.0,0.010356,...,0.070955,-0.010273,-0.468565,-0.428647,0.129794,-0.090424,NaN,-0.018947,0.065016,NaN
449,2025-07-08,-1.274646,-0.186248,-0.301696,-0.250000,0.75,NaN,0.0,0.0,-0.305203,...,-0.011313,-0.075198,-0.886455,-0.101630,-0.041632,-0.082992,NaN,-0.054519,0.065016,NaN
450,2025-11-18,-0.731982,-0.279759,-0.093511,-0.166667,-1.25,NaN,0.0,0.0,-0.463768,...,-0.036812,-0.029487,-0.502761,-0.068929,-0.121535,-0.082990,NaN,-0.034200,-0.144962,NaN
451,2025-11-25,0.435870,0.000000,0.279759,0.666667,3.00,NaN,0.0,0.0,-0.133333,...,0.039474,0.052470,0.322976,0.258088,-0.053041,-0.085009,NaN,0.031334,0.301242,NaN
